# Self-Optimising Hypergradients, Learned Optimisers, and Meta-Reweighting

### Historical Lineage & Theoretical Foundations
- **Hypergradient Descent (HD)** (*Baydin et al., 2018*): Differentiates validation/training loss with respect to optimizer hyperparameters online, updating learning rate via inner product $\Delta \eta = \beta \langle g_t, g_{t-1} \rangle$.
- **Meta-Sample Reweighting** (*Ren et al., 2018*): Evaluates per-sample training gradients against an unbiased validation gradient, zeroing out corrupted/noisy labels that exhibit negative cosine alignment.
- **Learned Optimizers** (*Andrychowicz et al., 2016; Metz et al., 2022*): Parameterizes optimization steps coordinate-wise via an MLP or recurrent cell $\Delta w = \text{MLP}_\phi([g_t, m_t])$, learning task-specific dynamics.

> [!WARNING]
> **Decision Boundary (When NOT to Use Unrolled Hypergradients):** Do not unroll meta-gradients over long optimization horizons ($T > 50$ steps) without implicit differentiation (IFT); meta-gradients explode exponentially at the Edge of Stability ($\rho(I - \eta H) > 1$). Always clamp hyper-learning-rate updates.

**Topic Connections:** `🔗 Upstream`: [Training Optimisations & Plasticity](02_training_optimisations_progressive_stacking_data_pruning_mrl_and_plasticity.ipynb) | `🔗 Downstream`: [Test-Time Adaptation & Non-Backprop](04_test_time_adaptation_and_learning.ipynb)


## 2. Mathematical Foundations & Governing Equations

1. **Hypergradient Descent Step (Baydin et al., 2018):** Online learning rate adaptation:
$$\nabla_{\eta} \mathcal{L}(w_{t+1}) = - \langle g_{t+1}, g_t \rangle \implies \eta_{t+1} = \eta_t + \beta \langle g_{t+1}, g_t \rangle, \quad w_{t+1} = w_t - \eta_{t+1} g_t$$
2. **Meta-Sample Reweighting (Ren et al., 2018):** Alignment with clean validation gradient $g_\text{val}$:
$$s_i = \langle G_{i, :}, g_\text{val} \rangle, \quad v_i = \max(0, s_i), \quad \bar{v}_i = \frac{v_i}{\sum_{k=1}^B v_k + \epsilon}, \quad \tilde{g} = \sum_{i=1}^B \bar{v}_i G_{i, :}$$
3. **Coordinate-wise Learned Optimizer MLP (Andrychowicz et al., 2016):**
$$\Delta w_j = \text{MLP}_\phi([g_j, m_j]), \quad w_j' = w_j - \Delta w_j, \quad m_j' = 0.9 m_j + 0.1 g_j$$
4. **Asymptotic Complexity:** HD step $\mathcal{O}(D)$; Meta-reweighting $\mathcal{O}(B D)$; Learned MLP step $\mathcal{O}(D)$.


In [ ]:
import sys
from pathlib import Path
import numpy as np, torch, torch.nn as nn

repo_root = Path.cwd().resolve()
while not (repo_root / "core").exists() and repo_root.parent != repo_root:
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
import core
core.setup_env(seed=42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ckpt_path = repo_root / "weights/04_training_methods_and_optimisers/03_training_dynamics_self_optimising_non_backprop/03_self_optimising_hypergradients_learned_optimisers_and_meta_reweighting.safetensors"
weights, test_in, test_out, meta = core.parity.load_reference_checkpoint(ckpt_path, device=device)
print(f"Device: {device} | Checkpoint: {ckpt_path.name} | Loaded {len(weights)} tensors")


In [ ]:
class LinearAnchor(nn.Module):
    def __init__(self, in_features: int = 8, out_features: int = 4):
        super().__init__()
        self.fc = nn.Linear(in_features, out_features)
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc(x)

def step_hypergradient_descent(w: torch.Tensor, g_curr: torch.Tensor, g_prev: torch.Tensor, lr: torch.Tensor, hyper_lr: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    h_t = torch.dot(g_curr, g_prev)
    lr_next = lr + hyper_lr * h_t
    return w - lr_next * g_curr, lr_next

def meta_sample_reweight(per_sample_grads: torch.Tensor, val_grad: torch.Tensor, eps: float = 1e-8) -> tuple[torch.Tensor, torch.Tensor]:
    alignments = per_sample_grads @ val_grad
    weights_relu = torch.clamp(alignments, min=0.0)
    norm_weights = weights_relu / (torch.sum(weights_relu) + eps)
    return norm_weights, norm_weights @ per_sample_grads

def step_learned_optimizer(param: torch.Tensor, grad: torch.Tensor, momentum: torch.Tensor, w1: torch.Tensor, b1: torch.Tensor, w2: torch.Tensor, b2: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    coords = torch.stack([grad, momentum], dim=-1)
    h = torch.relu(coords @ w1 + b1)
    delta_w = (h @ w2 + b2).squeeze(-1)
    return param - delta_w, 0.9 * momentum + 0.1 * grad, delta_w
print("Hypergradient & meta-optimization routines defined successfully.")


In [ ]:
anchor = LinearAnchor(8, 4).to(device)
anchor.fc.weight.data.copy_(weights['anchor_w'].T)
anchor.fc.bias.data.copy_(weights['anchor_b'])

w_hd, lr_hd = step_hypergradient_descent(weights['w_hd'], weights['g_curr'], weights['g_prev'], weights['lr_init'], weights['hyper_lr'])
err_lr = float(torch.max(torch.abs(lr_hd - weights['golden_lr_next'])).item())
err_w_hd = float(torch.max(torch.abs(w_hd - weights['golden_w_hd'])).item())
err_hd = max(err_lr, err_w_hd)

sample_w, rew_g = meta_sample_reweight(weights['per_sample_grads'], weights['val_grad'])
err_reweight = max(float(torch.max(torch.abs(sample_w - weights['golden_sample_weights'])).item()),
                   float(torch.max(torch.abs(rew_g - weights['golden_reweighted_grad'])).item()))

p_l, m_l, dw_l = step_learned_optimizer(weights['param_learned'], weights['grad_learned'], weights['momentum_learned'], weights['mlp_w1'], weights['mlp_b1'], weights['mlp_w2'], weights['mlp_b2'])
err_learned = max(float(torch.max(torch.abs(p_l - weights['golden_param_learned'])).item()),
                  float(torch.max(torch.abs(m_l - weights['golden_momentum_learned'])).item()),
                  float(torch.max(torch.abs(dw_l - weights['golden_delta_w'])).item()))

anchor_out = anchor(weights['anchor_x'])
err_anchor = float(torch.max(torch.abs(anchor_out - weights['golden_anchor_out'])).item())

for name, err in [('Hypergradient Descent', err_hd), ('Meta-Reweight', err_reweight), ('Learned MLP', err_learned), ('Anchor', err_anchor)]:
    assert err < 1e-5, f'{name} invariant failed: {err:.2e}'
print(f'All 4 invariants verified! HD: {err_hd:.2e}, Reweight: {err_reweight:.2e}, Learned: {err_learned:.2e}, Anchor: {err_anchor:.2e}')


In [ ]:
track_fns = {
    'Linear Anchor': lambda: anchor(weights['anchor_x']),
    'Hypergradient Step': lambda: step_hypergradient_descent(weights['w_hd'], weights['g_curr'], weights['g_prev'], weights['lr_init'], weights['hyper_lr']),
    'Meta-Sample Reweight': lambda: meta_sample_reweight(weights['per_sample_grads'], weights['val_grad']),
    'Learned Optimizer Step': lambda: step_learned_optimizer(weights['param_learned'], weights['grad_learned'], weights['momentum_learned'], weights['mlp_w1'], weights['mlp_b1'], weights['mlp_w2'], weights['mlp_b2']),
}
bench_results = core.benchmark.compare_tracks(track_fns, warmup=3, iters=15, batch_size=4)
print(core.benchmark.format_benchmark_table(bench_results))


In [ ]:
# Pathological Anti-Pattern: Uniform sample averaging when batch contains corrupted label noise
# Sample 0 is clean; Sample 1 is corrupted with an inverted gradient opposing validation direction.
g_clean = np.array([1.0, 1.0, 0.5])
g_noisy = np.array([-3.0, -3.0, -1.5]) # Corrupted label
g_val = np.array([1.0, 1.0, 0.5])

batch_g = np.stack([g_clean, g_noisy])
naive_grad = np.mean(batch_g, axis=0) # Corrupted sample overwhelms clean signal!
naive_cos = float(np.dot(naive_grad, g_val) / (np.linalg.norm(naive_grad) * np.linalg.norm(g_val)))

# Corrected Meta-Reweighting: zero weight to negative alignment
aligns = batch_g @ g_val
meta_w = np.maximum(0.0, aligns) / (np.sum(np.maximum(0.0, aligns)) + 1e-8)
corrected_grad = meta_w @ batch_g
corrected_cos = float(np.dot(corrected_grad, g_val) / (np.linalg.norm(corrected_grad) * np.linalg.norm(g_val)))

print(f'[FAILURE DEMONSTRATED] Naive batch gradient alignment with validation: cos={naive_cos:.2f} (INVERTED!)')
print(f'[CORRECT] Meta-Reweighting alignment with validation: cos={corrected_cos:.2f} (PERFECT 1.0)!')
assert naive_cos < 0.0 and corrected_cos > 0.99, 'Meta-reweighting failure contrast failed.'
print('[REMEDY] Use validation-loss gradient alignment to suppress corrupted gradients to zero weight.')


## 8. Algorithmic Takeaways & Cross-Domain Isomorphisms

- **Takeaways**: Hypergradient descent removes manual LR tuning by exploiting gradient trajectory curvature; meta-sample reweighting provides provable resilience to label noise; learned optimizers discover non-linear dynamics.

| Scientific Domain | Hypergradient Adaptation | Meta-Sample Reweighting | Learned Optimizer MLP |
| :--- | :--- | :--- | :--- |
| **Federated Learning** | Client-adaptive local learning rates | Zero-weight Byzantine poisoned client updates | Coordinate-wise client drift preconditioning |
| **Rare Particle Physics** | Dynamic damping around collision resonance | Reweight detector background noise to zero | Learn coordinate-wise invariant mass step scales |
| **High-Throughput Genomics** | Auto-tune step size across variable sequencing depths | Suppress PCR amplification artifacts | Multi-task parameter updates across cell lines |

---
**Navigation:** [← Training Optimisations & Plasticity](02_training_optimisations_progressive_stacking_data_pruning_mrl_and_plasticity.ipynb) | [↑ Master Index](../../README.md) | [Test-Time Adaptation & Non-Backprop →](04_test_time_adaptation_and_learning.ipynb)
